# Simulation (No Mutation, Exact Global Search)

This notebook performs an **exact global search on the configured discrete grid**. Every candidate contract is scored on exact FE-based mean/variance before the final decision is made. It keeps the same output file structure as the recent no-mutation simulation workflow.

In [ ]:
# Exact global-search simulation notebook (no mutation)
# This version evaluates the exact FE-based utility of every candidate on the configured
# discrete grid before the final decision is made.
#
# Optimality guarantee:
# - Global optimality holds on the configured discrete strike-price / fixed-volume grids.
# - This is stronger than the earlier screened+verified notebook, which did not globally
#   re-rank the entire grid on exact FE.
#
# Practical note:
# - This notebook is intentionally slower. It computes exact FE arrays in candidate batches.
# - For large match banks, run a subset first to estimate runtime.

from __future__ import annotations

import json
import math
import re
import time
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Tuple

import numpy as np
import pandas as pd

# ============================================================
# Main folders
# ============================================================
match_folder_name = "Match files of original data"
sample_root_dir = "Simplified baseline realized samples"
match_table_file = "Seller_Buyer_Match_Table.xlsx"
match_table_sheet = "All_Matches"
code2_correlation_file = Path("Input data and files") / "Correlation" / "correlation_parameter_bank_long.csv"

# Output folder for all per-match CSV files and all-match summary CSV files.
output_dir = "Output files (Risk Neutral, No Mutation, Exact Global Search)"

# ============================================================
# Risk preferences / contract settings
# ============================================================
# Utility convention is kept consistent with your latest simulation notebook.
# Set lambda_s = lambda_b = 0 for risk-neutral runs.
lambda_s = 0.0
lambda_b = 0.0

# Contract-period As-Generated guarantee fraction.
# The penalty price is calibrated as mean buyer_lmp_in; no gamma scale is used.
availability_factor = 0.95

# ============================================================
# Optimization grids
# ============================================================
# Global optimality is guaranteed only on these configured discrete grids.
strike_prices = np.arange(0.0, 600.0, 5.0)
fixed_volumes = np.arange(0.0, 20000.0, 20.0)

# ============================================================
# Exact-search batching controls
# ============================================================
# These do not change the solution. They only control runtime / memory.
exact_search_nonfix_strike_chunk_size = 20
exact_search_fix_volume_chunk_size = 5
exact_search_fix_strike_chunk_size = 10

# Progress printing for the exact global search.
exact_search_verbose = True
exact_search_progress_every_fix_blocks = 100

# ============================================================
# Batch controls
# ============================================================
selected_match_ids = None
skip_existing_output = False
save_per_match_grid = False
save_replication_fe = True
save_best_fe_even_if_infeasible = True
save_timestamp_in_replication_fe = False
scenario_name = "Baseline__No_Mutation__Exact_Global_Search"

# ============================================================
# Category defaults used for the plotting-ready summary table
# ============================================================
shape_corr_low = -0.30
shape_corr_high = 0.30
basis_corr_low = -0.30
basis_corr_high = 0.30
level_balance_tolerance = 0.05

# ============================================================
# Optional timestamp fallback for FE reconstruction later
# ============================================================
simulation_start = "2025-01-01 00:00:00"

CONFIG = {
    "match_folder_name": match_folder_name,
    "sample_root_dir": sample_root_dir,
    "match_table_file": match_table_file,
    "match_table_sheet": match_table_sheet,
    "code2_correlation_file": str(code2_correlation_file),
    "output_dir": output_dir,
    "lambda_s": float(lambda_s),
    "lambda_b": float(lambda_b),
    "availability_factor": float(availability_factor),
    "strike_prices": np.asarray(strike_prices, dtype=float),
    "fixed_volumes": np.asarray(fixed_volumes, dtype=float),
    "exact_search_nonfix_strike_chunk_size": int(exact_search_nonfix_strike_chunk_size),
    "exact_search_fix_volume_chunk_size": int(exact_search_fix_volume_chunk_size),
    "exact_search_fix_strike_chunk_size": int(exact_search_fix_strike_chunk_size),
    "exact_search_verbose": bool(exact_search_verbose),
    "exact_search_progress_every_fix_blocks": int(exact_search_progress_every_fix_blocks),
    "selected_match_ids": None if selected_match_ids is None else [int(x) for x in selected_match_ids],
    "skip_existing_output": bool(skip_existing_output),
    "save_per_match_grid": bool(save_per_match_grid),
    "save_replication_fe": bool(save_replication_fe),
    "save_best_fe_even_if_infeasible": bool(save_best_fe_even_if_infeasible),
    "save_timestamp_in_replication_fe": bool(save_timestamp_in_replication_fe),
    "scenario_name": str(scenario_name),
    "shape_corr_low": float(shape_corr_low),
    "shape_corr_high": float(shape_corr_high),
    "basis_corr_low": float(basis_corr_low),
    "basis_corr_high": float(basis_corr_high),
    "level_balance_tolerance": float(level_balance_tolerance),
    "simulation_start": str(simulation_start),
}
CONFIG

In [ ]:
# ============================================================
# Submission dtype helpers
# ============================================================
# These wrappers reduce DataFrame memory use without changing the financial
# calculations: identifiers/counters are downcast, repeated labels become
# categoricals, and continuous numerical columns remain float64.
import numpy as np

_PD_READ_CSV = pd.read_csv
_PD_READ_EXCEL = pd.read_excel

_INTEGER_DTYPE_CANDIDATES = {
    "match_id": np.int32,
    "hour": np.int16,
    "hour_index": np.int16,
    "replication": np.int16,
    "case_order": np.int16,
    "enabled": np.int8,
    "rank": np.int32,
}

_CATEGORY_DTYPE_CANDIDATES = {
    "case_id",
    "case_family",
    "case_label",
    "combined_category",
    "metric",
    "mutation_axis",
    "mutation_direction",
    "mutation_family",
    "mutation_label",
    "ppa_type",
    "profile_type",
    "risk_group",
    "risk_label",
    "scenario_name",
    "scenario_type",
    "solution_type",
    "status",
    "variable",
    "var_i",
    "var_j",
}


def _integer_dtype_fits(values, dtype) -> bool:
    if len(values) == 0:
        return True
    info = np.iinfo(dtype)
    return float(np.nanmin(values)) >= info.min and float(np.nanmax(values)) <= info.max


def optimize_dataframe_dtypes(df: pd.DataFrame) -> pd.DataFrame:
    """Conservatively compact non-financial columns after file loading."""
    if not isinstance(df, pd.DataFrame) or df.empty:
        return df

    for col in df.columns:
        series = df[col]
        if pd.api.types.is_integer_dtype(series.dtype):
            df[col] = pd.to_numeric(series, downcast="integer")

    for col, dtype in _INTEGER_DTYPE_CANDIDATES.items():
        if col not in df.columns:
            continue
        numeric = pd.to_numeric(df[col], errors="coerce")
        if numeric.isna().any():
            continue
        values = numeric.to_numpy(dtype="float64", copy=False)
        rounded = np.rint(values)
        if np.array_equal(values, rounded) and _integer_dtype_fits(rounded, dtype):
            df[col] = rounded.astype(dtype, copy=False)

    n_rows = len(df)
    for col in _CATEGORY_DTYPE_CANDIDATES.intersection(df.columns):
        series = df[col]
        if pd.api.types.is_categorical_dtype(series.dtype):
            continue
        if not (pd.api.types.is_object_dtype(series.dtype) or pd.api.types.is_string_dtype(series.dtype)):
            continue
        non_null = series.dropna()
        if non_null.empty:
            continue
        n_unique = int(non_null.nunique())
        if n_unique <= min(128, max(2, n_rows // 2)):
            df[col] = series.astype("category")

    return df


def read_csv_optimized(*args, **kwargs) -> pd.DataFrame:
    return optimize_dataframe_dtypes(_PD_READ_CSV(*args, **kwargs))


def read_excel_optimized(*args, **kwargs) -> pd.DataFrame:
    return optimize_dataframe_dtypes(_PD_READ_EXCEL(*args, **kwargs))


In [ ]:
# ============================================================
# Path resolution and input loading
# ============================================================
NOTEBOOK_CWD = Path.cwd().resolve()
BUNDLE_ROOT = NOTEBOOK_CWD
WORKSPACE_FOLDER_NAME = "simulation_baseline"


def resolve_simulation_baseline_root() -> Path:
    """Resolve the simulation_baseline workspace used for outputs."""
    anchors = [NOTEBOOK_CWD, BUNDLE_ROOT]
    for anchor in anchors:
        anchor = Path(anchor).resolve()
        if anchor.name == WORKSPACE_FOLDER_NAME:
            return anchor
        for parent in anchor.parents:
            if parent.name == WORKSPACE_FOLDER_NAME:
                return parent
        child = anchor / WORKSPACE_FOLDER_NAME
        if child.exists() and child.is_dir():
            return child.resolve()
    return BUNDLE_ROOT


SIMULATION_BASELINE_ROOT = resolve_simulation_baseline_root()


def candidate_roots(*anchors, max_parent_depth: int = 4):
    roots = []
    seen = set()
    for anchor in anchors:
        if anchor is None:
            continue
        p = Path(anchor).expanduser()
        if p.suffix:
            p = p.parent
        for root in [p, *list(p.parents)[:max_parent_depth]]:
            key = str(root)
            if key not in seen:
                seen.add(key)
                roots.append(root)
    return roots


def resolve_existing_dir(*candidates) -> Path:
    checked = []
    for candidate in candidates:
        if candidate is None:
            continue
        path = Path(candidate).expanduser()
        checked.append(path)
        if path.exists() and path.is_dir():
            return path.resolve()
    raise FileNotFoundError(
        "Could not resolve an existing directory. Tried:\n" +
        "\n".join(str(p) for p in checked)
    )


def resolve_existing_file(*candidates) -> Optional[Path]:
    checked = []
    for candidate in candidates:
        if candidate is None:
            continue
        path = Path(candidate).expanduser()
        checked.append(path)
        if path.exists() and path.is_file():
            return path.resolve()
    return None


def resolve_match_dir(match_folder_name: str) -> Path:
    roots = candidate_roots(BUNDLE_ROOT, NOTEBOOK_CWD, Path("/mnt/data"))
    candidates = []
    for root in roots:
        candidates.extend([
            root / match_folder_name,
            root / "Input data and files" / match_folder_name,
            root / "Data and files" / match_folder_name,
        ])
    return resolve_existing_dir(*candidates)


def resolve_sample_root(sample_root_dir: str, match_dir: Path) -> Path:
    roots = candidate_roots(BUNDLE_ROOT, NOTEBOOK_CWD, match_dir.parent, Path("/mnt/data"))
    candidates = []
    for root in roots:
        candidates.extend([
            root / sample_root_dir,
            root / "Input data and files" / sample_root_dir,
            root / "Data and files" / sample_root_dir,
        ])
    return resolve_existing_dir(*candidates)


def resolve_match_table(match_table_file: str) -> Optional[Path]:
    roots = candidate_roots(BUNDLE_ROOT, NOTEBOOK_CWD, Path("/mnt/data"))
    candidates = []
    for root in roots:
        candidates.extend([
            root / match_table_file,
            root / "Input data and files" / match_table_file,
            root / "Data and files" / match_table_file,
        ])
    return resolve_existing_file(*candidates)


def resolve_code2_corr_file(raw_path: str, match_dir: Path) -> Optional[Path]:
    candidate = Path(raw_path)
    roots = candidate_roots(BUNDLE_ROOT, NOTEBOOK_CWD, match_dir.parent, Path("/mnt/data"))
    candidates = []
    for root in roots:
        candidates.append(root / candidate)
    return resolve_existing_file(*candidates)


def resolve_output_root_dir(raw_path: str | Path) -> Path:
    candidate = Path(raw_path).expanduser()
    if candidate.is_absolute():
        return candidate.resolve()
    return (SIMULATION_BASELINE_ROOT / candidate).resolve()


def extract_match_id(path_like) -> int:
    path = Path(path_like)
    matched = re.search(r"(\d+)", path.stem)
    if matched is None:
        raise ValueError(f"Could not parse match_id from file name: {path.name}")
    return int(matched.group(1))


def load_match_table(match_table_path: Optional[Path], sheet_name: str = "All_Matches") -> pd.DataFrame:
    if match_table_path is None or not Path(match_table_path).exists():
        return pd.DataFrame()

    xls = pd.ExcelFile(match_table_path)
    if sheet_name in xls.sheet_names:
        df = read_excel_optimized(match_table_path, sheet_name=sheet_name)
    else:
        df = None
        for sh in xls.sheet_names:
            candidate = read_excel_optimized(match_table_path, sheet_name=sh)
            candidate.columns = [str(c).strip() for c in candidate.columns]
            if "match_id" in candidate.columns:
                df = candidate
                break
        if df is None:
            raise KeyError(f"Could not find a sheet with a 'match_id' column in {match_table_path}.")
    df.columns = [str(c).strip() for c in df.columns]
    if "match_id" in df.columns:
        df["match_id"] = pd.to_numeric(df["match_id"], errors="coerce").astype("Int64")
        df = df.dropna(subset=["match_id"]).copy()
        df["match_id"] = df["match_id"].astype(int)
    return df


def discover_match_ids(match_dir: Path, selected_match_ids: Optional[Iterable[int]] = None) -> List[int]:
    match_ids = sorted({extract_match_id(path) for path in match_dir.glob("*.csv")})
    if selected_match_ids is not None:
        selected = {int(x) for x in selected_match_ids}
        match_ids = [mid for mid in match_ids if mid in selected]
    if not match_ids:
        raise ValueError("No matches were found under the historical match folder.")
    return match_ids


def load_original_match_df(match_id: int, match_dir: Path) -> pd.DataFrame:
    path = match_dir / f"{int(match_id):03d}.csv"
    if not path.exists():
        alt = list(match_dir.glob(f"*{int(match_id):03d}*.csv"))
        if alt:
            path = alt[0]
    if not path.exists():
        raise FileNotFoundError(f"Historical match file was not found for match_id={match_id}.")
    df = read_csv_optimized(path)
    required = ["timestamp", "hour", "quarter", "generation", "demand", "seller_lmp", "buyer_lmp_out", "buyer_lmp_in"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise KeyError(f"{path.name} is missing required columns: {missing}")
    return df


def discover_replication_dirs(sample_root: Path) -> List[Tuple[int, Path]]:
    rep_dirs = []
    for path in sample_root.iterdir():
        if path.is_dir():
            matched = re.search(r"rep_(\d+)", path.name, flags=re.IGNORECASE)
            if matched:
                rep_dirs.append((int(matched.group(1)), path))
    rep_dirs = sorted(rep_dirs, key=lambda x: x[0])
    if not rep_dirs:
        raise FileNotFoundError(
            f"No replication subfolders like rep_01, rep_02, ... were found under {sample_root}."
        )
    return rep_dirs


def load_baseline_sample_bank(match_id: int, sample_root: Path) -> pd.DataFrame:
    frames = []
    rep_dirs = discover_replication_dirs(sample_root)

    for rep_no, rep_dir in rep_dirs:
        candidate = rep_dir / f"{int(match_id):03d}.csv"
        if not candidate.exists():
            alternatives = list(rep_dir.glob(f"*{int(match_id):03d}*.csv"))
            if alternatives:
                candidate = alternatives[0]
        if not candidate.exists():
            continue

        df = read_csv_optimized(candidate)
        required = ["timestamp", "hour", "quarter", "generation", "demand", "seller_lmp", "buyer_lmp_out", "buyer_lmp_in"]
        missing = [c for c in required if c not in df.columns]
        if missing:
            raise KeyError(f"{candidate} is missing required columns: {missing}")

        df = df.copy()
        df["replication"] = int(rep_no)
        df["hour_index"] = np.arange(1, len(df) + 1, dtype=int)
        df["sample_file"] = str(candidate)
        frames.append(df)

    if not frames:
        raise FileNotFoundError(
            f"No sample files were found for match_id={match_id} under {sample_root}."
        )

    bank = pd.concat(frames, ignore_index=True)
    bank["replication"] = pd.to_numeric(bank["replication"], errors="coerce").astype(int)
    bank["hour_index"] = pd.to_numeric(bank["hour_index"], errors="coerce").astype(int)
    return bank


def load_code2_corr_targets(code2_corr_path: Optional[Path]) -> pd.DataFrame:
    if code2_corr_path is None or not Path(code2_corr_path).exists():
        return pd.DataFrame(columns=["match_id", "var_i", "var_j", "corr_target"])

    df = read_csv_optimized(code2_corr_path)
    required = {"match_id", "var_i", "var_j"}
    if not required.issubset(df.columns):
        return pd.DataFrame(columns=["match_id", "var_i", "var_j", "corr_target"])

    if "corr_target" not in df.columns:
        if "corr_shrunk" in df.columns:
            df = df.rename(columns={"corr_shrunk": "corr_target"})
        elif "corr" in df.columns:
            df = df.rename(columns={"corr": "corr_target"})
        else:
            return pd.DataFrame(columns=["match_id", "var_i", "var_j", "corr_target"])

    df["match_id"] = pd.to_numeric(df["match_id"], errors="coerce").astype("Int64")
    df = df.dropna(subset=["match_id"]).copy()
    df["match_id"] = df["match_id"].astype(int)
    return df[["match_id", "var_i", "var_j", "corr_target"]].copy()


In [ ]:
# ============================================================
# Summary statistics, categories, and exact FE logic
# ============================================================
def safe_numeric(series) -> pd.Series:
    return pd.to_numeric(pd.Series(series), errors="coerce")


def safe_corr(a, b) -> float:
    a = safe_numeric(a)
    b = safe_numeric(b)
    valid = a.notna() & b.notna()
    if int(valid.sum()) < 2:
        return np.nan
    a_valid = a.loc[valid]
    b_valid = b.loc[valid]
    if a_valid.nunique() <= 1 or b_valid.nunique() <= 1:
        return np.nan
    return float(a_valid.corr(b_valid))


def summarize_series(series, prefix: str) -> Dict[str, float]:
    s = safe_numeric(series).dropna()
    if s.empty:
        return {
            f"{prefix}_mean": np.nan,
            f"{prefix}_median": np.nan,
            f"{prefix}_std": np.nan,
            f"{prefix}_q05": np.nan,
            f"{prefix}_q95": np.nan,
            f"{prefix}_min": np.nan,
            f"{prefix}_max": np.nan,
        }
    return {
        f"{prefix}_mean": float(s.mean()),
        f"{prefix}_median": float(s.median()),
        f"{prefix}_std": float(s.std(ddof=0)),
        f"{prefix}_q05": float(s.quantile(0.05)),
        f"{prefix}_q95": float(s.quantile(0.95)),
        f"{prefix}_min": float(s.min()),
        f"{prefix}_max": float(s.max()),
    }


def scalar_signed_index(left, right, use_abs_denominator: bool = False) -> float:
    left = float(left) if pd.notna(left) else np.nan
    right = float(right) if pd.notna(right) else np.nan
    if pd.isna(left) or pd.isna(right):
        return np.nan
    denominator = abs(left) + abs(right) if use_abs_denominator else left + right
    if denominator == 0:
        return np.nan
    return float((left - right) / denominator)


def classify_corr_regime(value: float, low: float, high: float,
                         label_high: str, label_mid: str, label_low: str) -> Tuple[float, Optional[str]]:
    if pd.isna(value):
        return np.nan, None
    if value >= high:
        return 1, label_high
    if value >= low:
        return 2, label_mid
    return 3, label_low


def classify_level_category(value: float, tol: float,
                            label_low: str, label_mid: str, label_high: str) -> Tuple[float, Optional[str]]:
    if pd.isna(value):
        return np.nan, None
    if value < -tol:
        return 1, label_low
    if value <= tol:
        return 2, label_mid
    return 3, label_high


def choose_preferred_value(original_value, simulated_value):
    if pd.notna(original_value):
        return original_value, "original"
    if pd.notna(simulated_value):
        return simulated_value, "simulated"
    return np.nan, None


def unusual_contracted_volume_flag(solution_row: pd.Series, sample_df: pd.DataFrame) -> str:
    q_opt = pd.to_numeric(pd.Series([solution_row.get("volume_mw", np.nan)]), errors="coerce").iloc[0]
    if pd.isna(q_opt) or q_opt <= 0:
        return "No"
    g95 = float(pd.to_numeric(sample_df["generation"], errors="coerce").quantile(0.95))
    d95 = float(pd.to_numeric(sample_df["demand"], errors="coerce").quantile(0.95))
    return "Yes" if (q_opt > 2.0 * g95 and q_opt > 2.0 * d95) else "No"


def build_basic_stats(match_id: int,
                      original_df: pd.DataFrame,
                      sample_df: pd.DataFrame,
                      corr_targets: pd.DataFrame,
                      config: Dict) -> Dict[str, object]:
    out: Dict[str, object] = {
        "match_id": int(match_id),
        "n_original_rows": int(len(original_df)),
        "n_sample_rows": int(len(sample_df)),
        "n_replications": int(sample_df["replication"].nunique()),
        "hours_per_replication": int(sample_df.groupby("replication")["hour_index"].max().median()),
    }

    original_map = {
        "generation": "generation",
        "demand": "demand",
        "seller_lmp": "seller_lmp",
        "buyer_lmp_out": "buyer_lmp_out",
        "buyer_lmp_in": "buyer_lmp_in",
    }
    simulated_map = original_map.copy()

    for var, col in original_map.items():
        out.update(summarize_series(original_df[col], f"{var}_original"))
    for var, col in simulated_map.items():
        out.update(summarize_series(sample_df[col], f"{var}_simulated"))

    out["shape_corr_original"] = safe_corr(original_df["generation"], original_df["demand"])
    out["basis_corr_original"] = safe_corr(original_df["seller_lmp"], original_df["buyer_lmp_out"])
    out["shape_corr_simulated"] = safe_corr(sample_df["generation"], sample_df["demand"])
    out["basis_corr_simulated"] = safe_corr(sample_df["seller_lmp"], sample_df["buyer_lmp_out"])

    def _lookup_target(match_id: int, v1: str, v2: str) -> float:
        if corr_targets.empty:
            return np.nan
        mask = (
            (corr_targets["match_id"].astype(int) == int(match_id)) &
            (
                ((corr_targets["var_i"] == v1) & (corr_targets["var_j"] == v2)) |
                ((corr_targets["var_i"] == v2) & (corr_targets["var_j"] == v1))
            )
        )
        sub = corr_targets.loc[mask, "corr_target"]
        if sub.empty:
            return np.nan
        return float(pd.to_numeric(sub, errors="coerce").dropna().iloc[0])

    out["shape_corr_target"] = _lookup_target(match_id, "generation", "demand")
    out["basis_corr_target"] = _lookup_target(match_id, "seller_lmp", "buyer_lmp_out")

    for metric in ["shape_corr", "basis_corr"]:
        val, src = choose_preferred_value(out.get(f"{metric}_original", np.nan), out.get(f"{metric}_simulated", np.nan))
        out[f"{metric}_ref"] = val
        out[f"{metric}_ref_source"] = src

    for metric in ["generation", "demand", "seller_lmp", "buyer_lmp_out"]:
        for stat in ["mean", "median"]:
            val, src = choose_preferred_value(out.get(f"{metric}_original_{stat}", np.nan), out.get(f"{metric}_simulated_{stat}", np.nan))
            out[f"{metric}_{stat}_ref"] = val
            out[f"{metric}_{stat}_ref_source"] = src

    out["volume_mismatch_mean_original"] = scalar_signed_index(
        out["generation_original_mean"], out["demand_original_mean"], use_abs_denominator=False
    )
    out["price_spread_mean_original"] = scalar_signed_index(
        out["seller_lmp_original_mean"], out["buyer_lmp_out_original_mean"], use_abs_denominator=True
    )
    out["volume_mismatch_median_original"] = scalar_signed_index(
        out["generation_original_median"], out["demand_original_median"], use_abs_denominator=False
    )
    out["price_spread_median_original"] = scalar_signed_index(
        out["seller_lmp_original_median"], out["buyer_lmp_out_original_median"], use_abs_denominator=True
    )

    out["volume_mismatch_mean_simulated"] = scalar_signed_index(
        out["generation_simulated_mean"], out["demand_simulated_mean"], use_abs_denominator=False
    )
    out["price_spread_mean_simulated"] = scalar_signed_index(
        out["seller_lmp_simulated_mean"], out["buyer_lmp_out_simulated_mean"], use_abs_denominator=True
    )
    out["volume_mismatch_median_simulated"] = scalar_signed_index(
        out["generation_simulated_median"], out["demand_simulated_median"], use_abs_denominator=False
    )
    out["price_spread_median_simulated"] = scalar_signed_index(
        out["seller_lmp_simulated_median"], out["buyer_lmp_out_simulated_median"], use_abs_denominator=True
    )

    for metric in ["volume_mismatch_mean", "price_spread_mean", "volume_mismatch_median", "price_spread_median"]:
        val, src = choose_preferred_value(out.get(f"{metric}_original", np.nan), out.get(f"{metric}_simulated", np.nan))
        out[f"{metric}_ref"] = val
        out[f"{metric}_ref_source"] = src

    out["shape_regime"], out["shape_regime_label"] = classify_corr_regime(
        out["shape_corr_ref"],
        config["shape_corr_low"],
        config["shape_corr_high"],
        "Coincident profile",
        "Weakly coupled profile",
        "Mismatched profile",
    )
    out["basis_regime"], out["basis_regime_label"] = classify_corr_regime(
        out["basis_corr_ref"],
        config["basis_corr_low"],
        config["basis_corr_high"],
        "Convergent market",
        "Moderately divergent market",
        "Highly divergent market",
    )
    out["volume_mean_cat"], out["volume_mean_cat_label"] = classify_level_category(
        out["volume_mismatch_mean_ref"],
        config["level_balance_tolerance"],
        "Seller < Buyer",
        "Balanced",
        "Seller > Buyer",
    )
    out["price_mean_cat"], out["price_mean_cat_label"] = classify_level_category(
        out["price_spread_mean_ref"],
        config["level_balance_tolerance"],
        "Seller < Buyer",
        "Balanced",
        "Seller > Buyer",
    )
    out["volume_median_cat"], out["volume_median_cat_label"] = classify_level_category(
        out["volume_mismatch_median_ref"],
        config["level_balance_tolerance"],
        "Seller < Buyer",
        "Balanced",
        "Seller > Buyer",
    )
    out["price_median_cat"], out["price_median_cat_label"] = classify_level_category(
        out["price_spread_median_ref"],
        config["level_balance_tolerance"],
        "Seller < Buyer",
        "Balanced",
        "Seller > Buyer",
    )

    out["combined_category"] = (
        f"S{int(out['shape_regime']) if pd.notna(out['shape_regime']) else 'NA'}_"
        f"B{int(out['basis_regime']) if pd.notna(out['basis_regime']) else 'NA'}_"
        f"Vm{int(out['volume_mean_cat']) if pd.notna(out['volume_mean_cat']) else 'NA'}_"
        f"Pm{int(out['price_mean_cat']) if pd.notna(out['price_mean_cat']) else 'NA'}_"
        f"Vd{int(out['volume_median_cat']) if pd.notna(out['volume_median_cat']) else 'NA'}_"
        f"Pd{int(out['price_median_cat']) if pd.notna(out['price_median_cat']) else 'NA'}"
    )

    return out


def compute_penalty_rate(sample_df: pd.DataFrame) -> Tuple[float, float]:
    """Calibrate the As-Generated contract-period penalty price.

    The revised formulation uses a fixed penalty price equal to the estimated
    contract-period average buyer purchase price, not a gamma-scaled tail
    quantile. The first returned value is the penalty price used in settlements;
    the second is retained as a named calibration diagnostic.
    """
    buyer_purchase_mean = float(pd.to_numeric(sample_df["buyer_lmp_in"], errors="coerce").mean())
    return buyer_purchase_mean, buyer_purchase_mean


def _replication_starts_from_ids(replication_ids) -> np.ndarray:
    ids = np.asarray(replication_ids)
    if ids.size == 0:
        raise ValueError("No replication observations are available.")
    changes = np.ones(ids.size, dtype=bool)
    changes[1:] = ids[1:] != ids[:-1]
    return np.flatnonzero(changes).astype(int)


def _sum_by_replication(values, replication_starts: np.ndarray) -> np.ndarray:
    arr = np.asarray(values, dtype=float)
    starts = np.asarray(replication_starts, dtype=int)
    if arr.shape[0] == 0:
        raise ValueError("Cannot aggregate an empty array by replication.")
    if starts.size == 0:
        raise ValueError("replication_starts is empty.")
    return np.add.reduceat(arr, starts, axis=0)


def _replication_ddof(n_replications: int) -> int:
    return 1 if int(n_replications) > 1 else 0


def _contract_period_asg_penalty_allocation(g, replication_starts: np.ndarray, availability_factor: float, penalty_rate: float) -> Tuple[np.ndarray, np.ndarray, float]:
    """Return row-level allocation of the once-per-replication AsG penalty.

    The penalty is computed on contract-period generation for each replication.
    For compatibility with existing row-level exports, the realized penalty is
    allocated to the first row of each replication; summing rows by replication
    exactly recovers the contract-period exposure.
    """
    g = np.asarray(g, dtype=float)
    generation_totals = _sum_by_replication(g, replication_starts)
    expected_generation_total = float(np.mean(generation_totals))
    penalty_by_replication = float(penalty_rate) * np.maximum(
        0.0,
        float(availability_factor) * expected_generation_total - generation_totals,
    )
    allocation = np.zeros_like(g, dtype=float)
    allocation[np.asarray(replication_starts, dtype=int)] = penalty_by_replication
    return allocation, penalty_by_replication, expected_generation_total


def _period_block_stats(fe_s_block, fe_b_block, rev_s_block, lambda_s: float, lambda_b: float, replication_starts: np.ndarray) -> Dict[str, np.ndarray]:
    seller_rep = _sum_by_replication(fe_s_block, replication_starts)
    buyer_rep = _sum_by_replication(fe_b_block, replication_starts)
    revenue_rep = _sum_by_replication(rev_s_block, replication_starts)
    ddof = _replication_ddof(seller_rep.shape[0])

    seller_mean = seller_rep.mean(axis=0)
    buyer_mean = buyer_rep.mean(axis=0)
    seller_var = seller_rep.var(axis=0, ddof=ddof)
    buyer_var = buyer_rep.var(axis=0, ddof=ddof)
    seller_std = np.sqrt(seller_var)
    buyer_std = np.sqrt(buyer_var)

    return {
        "seller_utility": seller_mean + float(lambda_s) * seller_std,
        "buyer_utility": buyer_mean + float(lambda_b) * buyer_std,
        "seller_mean_exposure": seller_mean,
        "buyer_mean_exposure": buyer_mean,
        "seller_exposure_variance": seller_var,
        "buyer_exposure_variance": buyer_var,
        "expected_seller_revenue": revenue_rep.mean(axis=0),
    }


def evaluate_contract(g, d, Ns, Nb_out, Nb_in, mu, nu, pi, q_fixed,
                      lambda_s, lambda_b, availability_factor, penalty_rate, replication_starts):
    T = len(g)

    if mu != "Physical":
        raise ValueError("Virtual PPA candidates are excluded by the revised physical-PPA formulation.")

    if nu == "Fix":
        q_del = np.full(T, q_fixed, dtype=float)
    elif nu == "AsG":
        q_del = np.asarray(g, dtype=float).copy()
    elif nu == "AsC":
        q_del = np.asarray(d, dtype=float).copy()
    else:
        raise ValueError(f"Unknown volume structure: {nu}")

    if nu == "AsG":
        Pen_t, _, _ = _contract_period_asg_penalty_allocation(
            g=g,
            replication_starts=replication_starts,
            availability_factor=availability_factor,
            penalty_rate=penalty_rate,
        )
    else:
        Pen_t = np.zeros(T, dtype=float)

    PPA_t = float(pi) * q_del
    f_s = Ns * (q_del - g) if nu in ["Fix", "AsC"] else np.zeros(T, dtype=float)
    f_b = Nb_in * np.maximum(0.0, d - q_del) - Nb_out * np.maximum(0.0, q_del - d)

    Rev_s = PPA_t - f_s - Pen_t
    FE_s = Ns * q_del - PPA_t + Pen_t
    FE_b = PPA_t + f_b - Nb_in * d - Pen_t

    stats = _summary_from_arrays(
        fe_s=FE_s,
        fe_b=FE_b,
        rev_s=Rev_s,
        lambda_s=lambda_s,
        lambda_b=lambda_b,
        replication_starts=replication_starts,
    )

    return {
        "q_del": q_del,
        "Pen_t": Pen_t,
        "PPA_t": PPA_t,
        "f_s": f_s,
        "f_b": f_b,
        "Rev_s": Rev_s,
        "FE_s": FE_s,
        "FE_b": FE_b,
        "U_s": stats["seller_utility"],
        "U_b": stats["buyer_utility"],
    }


def evaluate_no_contract(g, d, Ns, Nb_out, Nb_in, lambda_s, lambda_b, replication_starts):
    T = len(g)
    out = {
        "q_del": np.zeros(T, dtype=float),
        "Pen_t": np.zeros(T, dtype=float),
        "PPA_t": np.zeros(T, dtype=float),
        "f_s": -Ns * g,
        "f_b": Nb_in * d,
        "Rev_s": Ns * g,
        "FE_s": np.zeros(T, dtype=float),
        "FE_b": np.zeros(T, dtype=float),
    }
    stats = _summary_from_arrays(
        out["FE_s"], out["FE_b"], out["Rev_s"],
        lambda_s=lambda_s, lambda_b=lambda_b, replication_starts=replication_starts,
    )
    out["U_s"] = stats["seller_utility"]
    out["U_b"] = stats["buyer_utility"]
    return out


def _summary_from_arrays(fe_s, fe_b, rev_s, lambda_s, lambda_b, replication_starts: Optional[np.ndarray] = None) -> Dict[str, float]:
    if replication_starts is None:
        replication_starts = np.asarray([0], dtype=int)
    stats = _period_block_stats(
        fe_s_block=np.asarray(fe_s, dtype=float),
        fe_b_block=np.asarray(fe_b, dtype=float),
        rev_s_block=np.asarray(rev_s, dtype=float),
        lambda_s=lambda_s,
        lambda_b=lambda_b,
        replication_starts=replication_starts,
    )
    return {key: float(np.asarray(value)) for key, value in stats.items()}


def prepare_sample_arrays(sample_df: pd.DataFrame) -> Dict[str, object]:
    flat_df = sample_df.sort_values(["replication", "hour_index"]).reset_index(drop=True)
    replication_ids = pd.to_numeric(flat_df["replication"], errors="coerce").to_numpy()
    replication_starts = _replication_starts_from_ids(replication_ids)
    return {
        "flat_df": flat_df,
        "replication_ids": replication_ids,
        "replication_starts": replication_starts,
        "n_replications": int(len(replication_starts)),
        "g": pd.to_numeric(flat_df["generation"], errors="coerce").to_numpy(dtype=float),
        "d": pd.to_numeric(flat_df["demand"], errors="coerce").to_numpy(dtype=float),
        "Ns": pd.to_numeric(flat_df["seller_lmp"], errors="coerce").to_numpy(dtype=float),
        "Nb_out": pd.to_numeric(flat_df["buyer_lmp_out"], errors="coerce").to_numpy(dtype=float),
        "Nb_in": pd.to_numeric(flat_df["buyer_lmp_in"], errors="coerce").to_numpy(dtype=float),
    }


def evaluate_replication_fe_for_solution(sample_df: pd.DataFrame,
                                    solution: pd.Series,
                                    config: Dict,
                                    penalty_rate: float) -> pd.DataFrame:
    """Return one contract-period financial-exposure row per replication."""
    arrays = prepare_sample_arrays(sample_df)
    replication_starts = arrays["replication_starts"]
    replication_labels = arrays["flat_df"].iloc[replication_starts]["replication"].astype(int).to_numpy()
    ppa_type = str(solution.get("ppa_type", "No Contract"))
    profile_type = str(solution.get("profile_type", "N/A"))
    strike_price = pd.to_numeric(pd.Series([solution.get("strike_price_mwh", np.nan)]), errors="coerce").iloc[0]
    volume_mw = pd.to_numeric(pd.Series([solution.get("volume_mw", np.nan)]), errors="coerce").iloc[0]

    if ppa_type == "No Contract":
        out = evaluate_no_contract(
            arrays["g"], arrays["d"], arrays["Ns"], arrays["Nb_out"], arrays["Nb_in"],
            lambda_s=config["lambda_s"],
            lambda_b=config["lambda_b"],
            replication_starts=replication_starts,
        )
    else:
        out = evaluate_contract(
            g=arrays["g"],
            d=arrays["d"],
            Ns=arrays["Ns"],
            Nb_out=arrays["Nb_out"],
            Nb_in=arrays["Nb_in"],
            mu=ppa_type,
            nu=profile_type,
            pi=0.0 if pd.isna(strike_price) else float(strike_price),
            q_fixed=0.0 if pd.isna(volume_mw) else float(volume_mw),
            lambda_s=config["lambda_s"],
            lambda_b=config["lambda_b"],
            availability_factor=config["availability_factor"],
            penalty_rate=penalty_rate,
            replication_starts=replication_starts,
        )

    return pd.DataFrame({
        "replication": replication_labels,
        "seller_fe": _sum_by_replication(out["FE_s"], replication_starts),
        "buyer_fe": _sum_by_replication(out["FE_b"], replication_starts),
        "seller_revenue": _sum_by_replication(out["Rev_s"], replication_starts),
        "asg_shortfall_penalty": _sum_by_replication(out["Pen_t"], replication_starts),
    })




In [ ]:
# ============================================================
# Exact global search on the configured grid
# ============================================================
def _build_block_df_nonfix(ppa_type: str,
                           profile_type: str,
                           strike_prices_block: np.ndarray,
                           fe_s_block: np.ndarray,
                           fe_b_block: np.ndarray,
                           rev_s_block: np.ndarray,
                           lambda_s: float,
                           lambda_b: float,
                           replication_starts: np.ndarray) -> pd.DataFrame:
    stats = _period_block_stats(fe_s_block, fe_b_block, rev_s_block, lambda_s, lambda_b, replication_starts)
    return pd.DataFrame({
        "ppa_type": ppa_type,
        "profile_type": profile_type,
        "volume_mw": np.nan,
        "strike_price_mwh": strike_prices_block.astype(float),
        "seller_utility": np.asarray(stats["seller_utility"]).astype(float),
        "buyer_utility": np.asarray(stats["buyer_utility"]).astype(float),
        "seller_mean_exposure": np.asarray(stats["seller_mean_exposure"]).astype(float),
        "buyer_mean_exposure": np.asarray(stats["buyer_mean_exposure"]).astype(float),
        "seller_exposure_variance": np.asarray(stats["seller_exposure_variance"]).astype(float),
        "buyer_exposure_variance": np.asarray(stats["buyer_exposure_variance"]).astype(float),
        "expected_seller_revenue": np.asarray(stats["expected_seller_revenue"]).astype(float),
        "feasible": (np.asarray(stats["buyer_utility"]) <= 0.0),
        "decision_metric": "contract_period_global",
    })


def _build_block_df_fix(ppa_type: str,
                        volume_block: np.ndarray,
                        strike_prices_block: np.ndarray,
                        fe_s_block: np.ndarray,
                        fe_b_block: np.ndarray,
                        rev_s_block: np.ndarray,
                        lambda_s: float,
                        lambda_b: float,
                        replication_starts: np.ndarray) -> pd.DataFrame:
    stats = _period_block_stats(fe_s_block, fe_b_block, rev_s_block, lambda_s, lambda_b, replication_starts)
    vol_grid, strike_grid = np.meshgrid(volume_block.astype(float), strike_prices_block.astype(float), indexing="ij")
    return pd.DataFrame({
        "ppa_type": ppa_type,
        "profile_type": "Fix",
        "volume_mw": vol_grid.reshape(-1),
        "strike_price_mwh": strike_grid.reshape(-1),
        "seller_utility": np.asarray(stats["seller_utility"]).reshape(-1).astype(float),
        "buyer_utility": np.asarray(stats["buyer_utility"]).reshape(-1).astype(float),
        "seller_mean_exposure": np.asarray(stats["seller_mean_exposure"]).reshape(-1).astype(float),
        "buyer_mean_exposure": np.asarray(stats["buyer_mean_exposure"]).reshape(-1).astype(float),
        "seller_exposure_variance": np.asarray(stats["seller_exposure_variance"]).reshape(-1).astype(float),
        "buyer_exposure_variance": np.asarray(stats["buyer_exposure_variance"]).reshape(-1).astype(float),
        "expected_seller_revenue": np.asarray(stats["expected_seller_revenue"]).reshape(-1).astype(float),
        "feasible": (np.asarray(stats["buyer_utility"]).reshape(-1) <= 0.0),
        "decision_metric": "contract_period_global",
    })


def optimize_ppa_contracts_exact_global(sample_df: pd.DataFrame, config: Dict, penalty_rate: float) -> Tuple[pd.DataFrame, Dict[str, object]]:
    arrays = prepare_sample_arrays(sample_df)
    g = arrays["g"]
    d = arrays["d"]
    Ns = arrays["Ns"]
    Nb_out = arrays["Nb_out"]
    Nb_in = arrays["Nb_in"]
    replication_starts = arrays["replication_starts"]

    strike_prices = np.asarray(config["strike_prices"], dtype=float)
    fixed_volumes = np.asarray([float(q) for q in config["fixed_volumes"] if np.isfinite(q) and float(q) > 0.0], dtype=float)
    lambda_s = float(config["lambda_s"])
    lambda_b = float(config["lambda_b"])
    availability_factor = float(config["availability_factor"])

    nonfix_strike_chunk = max(1, int(config["exact_search_nonfix_strike_chunk_size"]))
    fix_volume_chunk = max(1, int(config["exact_search_fix_volume_chunk_size"]))
    fix_strike_chunk = max(1, int(config["exact_search_fix_strike_chunk_size"]))
    verbose = bool(config.get("exact_search_verbose", True))
    progress_every = max(1, int(config.get("exact_search_progress_every_fix_blocks", 100)))

    t0 = time.perf_counter()
    frames: List[pd.DataFrame] = []

    # Outside option
    no_contract = evaluate_no_contract(
        g, d, Ns, Nb_out, Nb_in,
        lambda_s=lambda_s,
        lambda_b=lambda_b,
        replication_starts=replication_starts,
    )
    no_contract_stats = _summary_from_arrays(
        no_contract["FE_s"], no_contract["FE_b"], no_contract["Rev_s"],
        lambda_s=lambda_s, lambda_b=lambda_b, replication_starts=replication_starts,
    )
    frames.append(pd.DataFrame([{
        "ppa_type": "No Contract",
        "profile_type": "N/A",
        "volume_mw": np.nan,
        "strike_price_mwh": np.nan,
        **no_contract_stats,
        "feasible": True,
        "decision_metric": "contract_period_global",
    }]))

    penalty_asg, penalty_by_replication, expected_generation_total = _contract_period_asg_penalty_allocation(
        g=g,
        replication_starts=replication_starts,
        availability_factor=availability_factor,
        penalty_rate=penalty_rate,
    )
    Ns_col = Ns[:, None]
    g_col = g[:, None]
    d_col = d[:, None]
    nb_in_col = Nb_in[:, None]
    nb_out_col = Nb_out[:, None]
    penalty_col = penalty_asg[:, None]
    revenue_no_contract = (Ns * g)[:, None]

    # ---------- AsG / AsC exact physical candidates ----------
    if verbose:
        print("  Exact search: non-Fix physical candidates ...")

    nonfix_pairs = [
        ("Physical", "AsG"),
        ("Physical", "AsC"),
    ]

    for ppa_type, profile_type in nonfix_pairs:
        for start in range(0, len(strike_prices), nonfix_strike_chunk):
            strike_block = strike_prices[start:start + nonfix_strike_chunk]
            pi_row = strike_block[None, :]

            if profile_type == "AsG":
                fe_s = g_col * (Ns_col - pi_row) + penalty_col
                fe_b = (
                    pi_row * g_col
                    + nb_in_col * np.maximum(d - g, 0.0)[:, None]
                    - nb_out_col * np.maximum(g - d, 0.0)[:, None]
                    - nb_in_col * d_col
                    - penalty_col
                )
                rev_s = pi_row * g_col - penalty_col

            elif profile_type == "AsC":
                fe_s = d_col * (Ns_col - pi_row)
                fe_b = d_col * (pi_row - nb_in_col)
                rev_s = revenue_no_contract + d_col * (pi_row - Ns_col)

            else:
                raise ValueError(f"Unsupported physical profile type: {profile_type}")

            frames.append(
                _build_block_df_nonfix(
                    ppa_type=ppa_type,
                    profile_type=profile_type,
                    strike_prices_block=strike_block,
                    fe_s_block=fe_s,
                    fe_b_block=fe_b,
                    rev_s_block=rev_s,
                    lambda_s=lambda_s,
                    lambda_b=lambda_b,
                    replication_starts=replication_starts,
                )
            )

    # ---------- Fixed-Volume exact physical candidates ----------
    if verbose:
        total_fix_blocks = (
            math.ceil(len(fixed_volumes) / fix_volume_chunk)
            * math.ceil(len(strike_prices) / fix_strike_chunk)
        )
        print(
            "  Exact search: Fixed-Volume physical candidates ... "
            f"{len(fixed_volumes) * len(strike_prices):,} candidates "
            f"in {total_fix_blocks:,} blocks"
        )

    Ns_3d = Ns[:, None, None]
    d_3d = d[:, None, None]
    nb_in_3d = Nb_in[:, None, None]
    nb_out_3d = Nb_out[:, None, None]
    revenue_no_contract_3d = (Ns * g)[:, None, None]

    block_counter = 0
    for v_start in range(0, len(fixed_volumes), fix_volume_chunk):
        volume_block = fixed_volumes[v_start:v_start + fix_volume_chunk]
        q_3d = volume_block[None, :, None]

        for p_start in range(0, len(strike_prices), fix_strike_chunk):
            strike_block = strike_prices[p_start:p_start + fix_strike_chunk]
            pi_3d = strike_block[None, None, :]

            fe_s = q_3d * (Ns_3d - pi_3d)
            fe_b = (
                q_3d * pi_3d
                + nb_in_3d * np.maximum(d_3d - q_3d, 0.0)
                - nb_out_3d * np.maximum(q_3d - d_3d, 0.0)
                - nb_in_3d * d_3d
            )
            rev_s = revenue_no_contract_3d + q_3d * (pi_3d - Ns_3d)

            frames.append(
                _build_block_df_fix(
                    ppa_type="Physical",
                    volume_block=volume_block,
                    strike_prices_block=strike_block,
                    fe_s_block=fe_s,
                    fe_b_block=fe_b,
                    rev_s_block=rev_s,
                    lambda_s=lambda_s,
                    lambda_b=lambda_b,
                    replication_starts=replication_starts,
                )
            )

            block_counter += 1
            if verbose and (block_counter % progress_every == 0):
                elapsed = time.perf_counter() - t0
                print(f"    processed {block_counter:,} Fix blocks | elapsed {elapsed:,.1f}s")

    grid_df = pd.concat(frames, ignore_index=True)
    elapsed = time.perf_counter() - t0

    search_report = {
        "decision_search_mode": "exact_global_grid",
        "exact_fe_evaluation_before_decision": True,
        "global_optimality_on_exact_grid": True,
        "virtual_ppa_candidates_evaluated": 0,
        "contract_period_penalty": True,
        "exposure_aggregation_level": "replication_contract_period_total",
        "n_replications": int(arrays["n_replications"]),
        "expected_generation_total_for_asg_penalty": float(expected_generation_total),
        "mean_asg_penalty_across_replications": float(np.mean(penalty_by_replication)),
        "verification_enabled": False,
        "formula_best_verification_passed": True,
        "formula_best_changed_after_verification": False,
        "formula_best_ppa_changed_after_verification": False,
        "n_exact_candidates_evaluated": int(len(grid_df)),
        "n_total_candidates": int(len(grid_df)),
        "n_total_ppa_candidates": int((grid_df["ppa_type"].astype(str) != "No Contract").sum()),
        "n_nonfix_candidates": int(((grid_df["profile_type"].astype(str) != "Fix") & (grid_df["ppa_type"].astype(str) != "No Contract")).sum()),
        "n_fix_candidates": int((grid_df["profile_type"].astype(str) == "Fix").sum()),
        "n_feasible_candidates": int(pd.Series(grid_df["feasible"]).fillna(False).sum()),
        "n_feasible_ppa_candidates": int(((grid_df["ppa_type"].astype(str) != "No Contract") & pd.Series(grid_df["feasible"]).fillna(False)).sum()),
        "exact_search_elapsed_seconds": float(elapsed),
        "exact_search_nonfix_strike_chunk_size": int(nonfix_strike_chunk),
        "exact_search_fix_volume_chunk_size": int(fix_volume_chunk),
        "exact_search_fix_strike_chunk_size": int(fix_strike_chunk),
        "exact_search_verbose": bool(verbose),
        "fix_repair_reports_json": json.dumps([]),
        "n_fix_seed_rows": 0,
    }
    return grid_df, search_report


def get_optimal_solution(df_results: pd.DataFrame) -> pd.Series:
    feasible_df = df_results.loc[df_results["feasible"] == True].copy()
    if feasible_df.empty:
        idx = df_results["buyer_utility"].idxmin()
    else:
        idx = feasible_df["seller_utility"].idxmin()
    out = df_results.loc[idx].copy()
    out["best_solution"] = True
    return out


def get_best_ppa_solution(df_results: pd.DataFrame) -> Optional[pd.Series]:
    ppa_df = df_results.loc[df_results["ppa_type"].astype(str) != "No Contract"].copy()
    if ppa_df.empty:
        return None
    feasible_ppa_df = ppa_df.loc[ppa_df["feasible"] == True].copy()
    if feasible_ppa_df.empty:
        idx = ppa_df["buyer_utility"].idxmin()
    else:
        idx = feasible_ppa_df["seller_utility"].idxmin()
    out = ppa_df.loc[idx].copy()
    out["best_ppa_solution"] = True
    return out


def select_replication_fe_export_solution(best_solution: pd.Series,
                                     best_ppa_solution: Optional[pd.Series],
                                     save_best_fe_even_if_infeasible: bool) -> Tuple[pd.Series, str]:
    if str(best_solution.get("ppa_type", "No Contract")) != "No Contract":
        return best_solution, "Selected_Best_Solution"

    if not save_best_fe_even_if_infeasible:
        return best_solution, "Selected_Best_Solution"

    if best_ppa_solution is None:
        return best_solution, "Selected_Best_Solution"

    if str(best_ppa_solution.get("ppa_type", "No Contract")) == "No Contract":
        return best_solution, "Selected_Best_Solution"

    if bool(best_ppa_solution.get("feasible", False)):
        return best_ppa_solution, "Best_Feasible_PPA_While_Decision_Is_No_Contract"

    return best_ppa_solution, "Best_Infeasible_PPA"


In [ ]:
# ============================================================
# Output writing and batch run
# ============================================================
def build_case_info_long(match_id: int,
                         match_row: Dict[str, object],
                         basic_stats: Dict[str, object],
                         penalty_rate: float,
                         buyer_purchase_mean: float,
                         best_solution: pd.Series,
                         best_ppa_solution: Optional[pd.Series],
                         search_report: Dict[str, object],
                         scenario_name: str,
                         config: Dict) -> pd.DataFrame:
    records = []

    def add(section: str, item: str, value):
        records.append({"section": section, "item": item, "value": value})

    add("Scenario_Bank", "scenario_name", scenario_name)
    add("Scenario_Bank", "sample_root_dir", config["sample_root_dir"])
    add("Scenario_Bank", "n_replications", basic_stats.get("n_replications"))
    add("Scenario_Bank", "hours_per_replication", basic_stats.get("hours_per_replication"))

    add("Risk_Setting", "lambda_s", config["lambda_s"])
    add("Risk_Setting", "lambda_b", config["lambda_b"])
    add("Risk_Setting", "availability_factor", config["availability_factor"])
    add("Risk_Setting", "penalty_rate", penalty_rate)
    add("Risk_Setting", "buyer_lmp_in_mean", buyer_purchase_mean)

    add("Decision", "ppa_type", best_solution.get("ppa_type"))
    add("Decision", "profile_type", best_solution.get("profile_type"))
    add("Decision", "strike_price_mwh", best_solution.get("strike_price_mwh"))
    add("Decision", "volume_mw", best_solution.get("volume_mw"))
    add("Decision", "seller_utility", best_solution.get("seller_utility"))
    add("Decision", "buyer_utility", best_solution.get("buyer_utility"))
    add("Decision", "feasible", best_solution.get("feasible"))

    if best_ppa_solution is not None:
        add("Best_PPA", "ppa_type", best_ppa_solution.get("ppa_type"))
        add("Best_PPA", "profile_type", best_ppa_solution.get("profile_type"))
        add("Best_PPA", "strike_price_mwh", best_ppa_solution.get("strike_price_mwh"))
        add("Best_PPA", "volume_mw", best_ppa_solution.get("volume_mw"))
        add("Best_PPA", "seller_utility", best_ppa_solution.get("seller_utility"))
        add("Best_PPA", "buyer_utility", best_ppa_solution.get("buyer_utility"))
        add("Best_PPA", "feasible", best_ppa_solution.get("feasible"))

    for key, value in search_report.items():
        add("Exact_Search", key, value)

    for key, value in match_row.items():
        add("Match_Metadata", key, value)

    stat_sections = {
        "Correlations": [
            "shape_corr_original", "basis_corr_original", "shape_corr_simulated", "basis_corr_simulated",
            "shape_corr_target", "basis_corr_target", "shape_corr_ref", "basis_corr_ref",
        ],
        "Means": [
            "generation_original_mean", "demand_original_mean", "seller_lmp_original_mean", "buyer_lmp_out_original_mean", "buyer_lmp_in_original_mean",
            "generation_simulated_mean", "demand_simulated_mean", "seller_lmp_simulated_mean", "buyer_lmp_out_simulated_mean", "buyer_lmp_in_simulated_mean",
        ],
        "Medians": [
            "generation_original_median", "demand_original_median", "seller_lmp_original_median", "buyer_lmp_out_original_median", "buyer_lmp_in_original_median",
            "generation_simulated_median", "demand_simulated_median", "seller_lmp_simulated_median", "buyer_lmp_out_simulated_median", "buyer_lmp_in_simulated_median",
        ],
        "Plot_Indices": [
            "volume_mismatch_mean_ref", "price_spread_mean_ref", "volume_mismatch_median_ref", "price_spread_median_ref",
        ],
        "Categories": [
            "shape_regime", "shape_regime_label", "basis_regime", "basis_regime_label",
            "volume_mean_cat", "volume_mean_cat_label", "price_mean_cat", "price_mean_cat_label",
            "volume_median_cat", "volume_median_cat_label", "price_median_cat", "price_median_cat_label",
            "combined_category",
        ],
    }

    for section, items in stat_sections.items():
        for item in items:
            add(section, item, basic_stats.get(item))

    return pd.DataFrame(records)


def to_plain_dict(series: Optional[pd.Series]) -> Dict[str, object]:
    if series is None:
        return {}
    data = {}
    for key, value in series.to_dict().items():
        if isinstance(value, (np.floating, np.integer)):
            data[key] = value.item()
        else:
            data[key] = value
    return data


def run_one_match_exact_global(match_id: int,
                               match_dir: Path,
                               sample_root: Path,
                               output_root: Path,
                               match_table_df: pd.DataFrame,
                               corr_targets: pd.DataFrame,
                               config: Dict) -> Dict[str, object]:
    original_df = load_original_match_df(match_id, match_dir)
    sample_df = load_baseline_sample_bank(match_id, sample_root)
    penalty_rate, buyer_purchase_mean = compute_penalty_rate(sample_df)

    grid_df, search_report = optimize_ppa_contracts_exact_global(
        sample_df=sample_df,
        config=config,
        penalty_rate=penalty_rate,
    )

    best_solution = get_optimal_solution(grid_df)
    best_ppa_solution = get_best_ppa_solution(grid_df)

    basic_stats = build_basic_stats(
        match_id=match_id,
        original_df=original_df,
        sample_df=sample_df,
        corr_targets=corr_targets,
        config=config,
    )

    if not match_table_df.empty and "match_id" in match_table_df.columns:
        selected = match_table_df.loc[match_table_df["match_id"] == int(match_id)].copy()
        match_row = selected.iloc[0].to_dict() if not selected.empty else {}
    else:
        match_row = {}

    selected_fe_solution, fe_source = select_replication_fe_export_solution(
        best_solution=best_solution,
        best_ppa_solution=best_ppa_solution,
        save_best_fe_even_if_infeasible=config["save_best_fe_even_if_infeasible"],
    )

    match_output_dir = output_root / "Per_Match" / f"Match_{int(match_id):04d}"
    match_output_dir.mkdir(parents=True, exist_ok=True)

    case_info_df = build_case_info_long(
        match_id=match_id,
        match_row=match_row,
        basic_stats=basic_stats,
        penalty_rate=penalty_rate,
        buyer_purchase_mean=buyer_purchase_mean,
        best_solution=best_solution,
        best_ppa_solution=best_ppa_solution,
        search_report=search_report,
        scenario_name=config["scenario_name"],
        config=config,
    )
    case_info_path = match_output_dir / "Case_Info.csv"
    case_info_df.to_csv(case_info_path, index=False)

    best_solution_dict = to_plain_dict(best_solution)
    best_solution_dict["match_id"] = int(match_id)
    best_solution_dict["scenario_name"] = config["scenario_name"]
    best_solution_dict["penalty_rate"] = penalty_rate
    best_solution_dict["buyer_lmp_in_mean"] = buyer_purchase_mean
    best_solution_dict["unusual_contracted_volume"] = unusual_contracted_volume_flag(best_solution, sample_df)
    # compatibility fields for existing downstream plotting / scripts
    best_solution_dict["formula_selected_ppa_type"] = best_solution.get("ppa_type")
    best_solution_dict["formula_selected_profile_type"] = best_solution.get("profile_type")
    best_solution_dict["formula_selected_volume_mw"] = best_solution.get("volume_mw")
    best_solution_dict["formula_selected_strike_price_mwh"] = best_solution.get("strike_price_mwh")
    best_solution_dict.update(search_report)
    best_solution_path = match_output_dir / "Best_Solution.csv"
    pd.DataFrame([best_solution_dict]).to_csv(best_solution_path, index=False)

    best_ppa_solution_dict = to_plain_dict(best_ppa_solution) if best_ppa_solution is not None else {}
    if best_ppa_solution_dict:
        best_ppa_solution_dict["match_id"] = int(match_id)
        best_ppa_solution_dict["scenario_name"] = config["scenario_name"]
        best_ppa_solution_dict["penalty_rate"] = penalty_rate
        best_ppa_solution_dict["buyer_lmp_in_mean"] = buyer_purchase_mean
        best_ppa_solution_dict["unusual_contracted_volume"] = unusual_contracted_volume_flag(best_ppa_solution, sample_df)
        # compatibility fields
        best_ppa_solution_dict["formula_best_ppa_type"] = best_ppa_solution.get("ppa_type")
        best_ppa_solution_dict["formula_best_ppa_profile_type"] = best_ppa_solution.get("profile_type")
        best_ppa_solution_dict["formula_best_ppa_volume_mw"] = best_ppa_solution.get("volume_mw")
        best_ppa_solution_dict["formula_best_ppa_strike_price_mwh"] = best_ppa_solution.get("strike_price_mwh")
        best_ppa_solution_dict.update(search_report)
        best_ppa_solution_path = match_output_dir / "Best_PPA_Solution.csv"
        pd.DataFrame([best_ppa_solution_dict]).to_csv(best_ppa_solution_path, index=False)
    else:
        best_ppa_solution_path = None

    if config["save_per_match_grid"]:
        grid_path = match_output_dir / "Contract_Grid.csv"
        grid_df.to_csv(grid_path, index=False)
    else:
        grid_path = None

    replication_fe_path = None
    replication_fe_index_row = None
    if config["save_replication_fe"]:
        fe_df = evaluate_replication_fe_for_solution(
            sample_df=sample_df,
            solution=selected_fe_solution,
            config=config,
            penalty_rate=penalty_rate,
        )
        fe_subdir = output_root / "Replication_FE"
        fe_subdir.mkdir(parents=True, exist_ok=True)
        replication_fe_path = fe_subdir / f"Replication_FE_Match_{int(match_id):04d}__{config['scenario_name']}.csv"
        fe_df.to_csv(replication_fe_path, index=False)

        replication_fe_index_row = {
            "match_id": int(match_id),
            "scenario_name": config["scenario_name"],
            "scenario_type": "baseline",
            "replication_fe_file": str(replication_fe_path),
            "fe_source": fe_source,
            "decision_ppa_type": best_solution.get("ppa_type"),
            "decision_profile_type": best_solution.get("profile_type"),
            "decision_strike_price_mwh": best_solution.get("strike_price_mwh"),
            "decision_volume_mw": best_solution.get("volume_mw"),
            "formula_decision_ppa_type": best_solution.get("ppa_type"),
            "formula_decision_profile_type": best_solution.get("profile_type"),
            "formula_decision_strike_price_mwh": best_solution.get("strike_price_mwh"),
            "formula_decision_volume_mw": best_solution.get("volume_mw"),
            "ppa_type": selected_fe_solution.get("ppa_type"),
            "profile_type": selected_fe_solution.get("profile_type"),
            "strike_price_mwh": selected_fe_solution.get("strike_price_mwh"),
            "volume_mw": selected_fe_solution.get("volume_mw"),
            "feasible": selected_fe_solution.get("feasible"),
            "n_replications": basic_stats["n_replications"],
            "hours_per_replication": basic_stats["hours_per_replication"],
            **search_report,
        }

    summary_row = dict(basic_stats)
    summary_row.update({
        "scenario_name": config["scenario_name"],
        "seller_utility": best_solution.get("seller_utility"),
        "buyer_utility": best_solution.get("buyer_utility"),
        "seller_mean_exposure": best_solution.get("seller_mean_exposure"),
        "buyer_mean_exposure": best_solution.get("buyer_mean_exposure"),
        "seller_exposure_variance": best_solution.get("seller_exposure_variance"),
        "buyer_exposure_variance": best_solution.get("buyer_exposure_variance"),
        "expected_seller_revenue": best_solution.get("expected_seller_revenue"),
        "ppa_type": best_solution.get("ppa_type"),
        "profile_type": best_solution.get("profile_type"),
        "volume_mw": best_solution.get("volume_mw"),
        "strike_price_mwh": best_solution.get("strike_price_mwh"),
        "feasible": best_solution.get("feasible"),
        "unusual_contracted_volume": unusual_contracted_volume_flag(best_solution, sample_df),
        "best_ppa_type": best_ppa_solution.get("ppa_type") if best_ppa_solution is not None else None,
        "best_ppa_profile_type": best_ppa_solution.get("profile_type") if best_ppa_solution is not None else None,
        "best_ppa_volume_mw": best_ppa_solution.get("volume_mw") if best_ppa_solution is not None else np.nan,
        "best_ppa_strike_price_mwh": best_ppa_solution.get("strike_price_mwh") if best_ppa_solution is not None else np.nan,
        "best_ppa_feasible": best_ppa_solution.get("feasible") if best_ppa_solution is not None else None,
        # compatibility fields
        "formula_selected_ppa_type": best_solution.get("ppa_type"),
        "formula_selected_profile_type": best_solution.get("profile_type"),
        "formula_selected_volume_mw": best_solution.get("volume_mw"),
        "formula_selected_strike_price_mwh": best_solution.get("strike_price_mwh"),
        "formula_best_ppa_type": best_ppa_solution.get("ppa_type") if best_ppa_solution is not None else None,
        "formula_best_ppa_profile_type": best_ppa_solution.get("profile_type") if best_ppa_solution is not None else None,
        "formula_best_ppa_volume_mw": best_ppa_solution.get("volume_mw") if best_ppa_solution is not None else np.nan,
        "formula_best_ppa_strike_price_mwh": best_ppa_solution.get("strike_price_mwh") if best_ppa_solution is not None else np.nan,
        "penalty_rate": penalty_rate,
        "buyer_lmp_in_mean": buyer_purchase_mean,
        "fe_source": fe_source,
        "case_info_file": str(case_info_path),
        "best_solution_file": str(best_solution_path),
        "best_ppa_solution_file": str(best_ppa_solution_path) if best_ppa_solution_path is not None else "",
        "contract_grid_file": str(grid_path) if grid_path is not None else "",
        "replication_fe_file": str(replication_fe_path) if replication_fe_path is not None else "",
        **search_report,
    })

    for key, value in match_row.items():
        summary_row[f"match__{key}"] = value

    return {
        "summary_row": summary_row,
        "best_solution_row": best_solution_dict,
        "best_ppa_solution_row": best_ppa_solution_dict,
        "replication_fe_index_row": replication_fe_index_row,
        "search_report": search_report,
    }


match_dir = resolve_match_dir(CONFIG["match_folder_name"])
sample_root = resolve_sample_root(CONFIG["sample_root_dir"], match_dir=match_dir)
match_table_path = resolve_match_table(CONFIG["match_table_file"])
match_table_df = load_match_table(match_table_path, sheet_name=CONFIG["match_table_sheet"])
corr_targets_path = resolve_code2_corr_file(CONFIG["code2_correlation_file"], match_dir=match_dir)
corr_targets_df = load_code2_corr_targets(corr_targets_path)

output_root = resolve_output_root_dir(CONFIG["output_dir"])
output_root.mkdir(parents=True, exist_ok=True)
(output_root / "Per_Match").mkdir(parents=True, exist_ok=True)
if CONFIG["save_replication_fe"]:
    (output_root / "Replication_FE").mkdir(parents=True, exist_ok=True)

match_ids = discover_match_ids(match_dir, selected_match_ids=CONFIG["selected_match_ids"])

print("Resolved paths:")
print("  Historical match folder :", match_dir)
print("  Sample root             :", sample_root)
print("  Match table             :", match_table_path if match_table_path is not None else "Not found (optional)")
print("  Code 2 corr file        :", corr_targets_path if corr_targets_path is not None else "Not found (optional)")
print("  Output root             :", output_root)
print()
print(f"Matches selected: {len(match_ids)}")
print(f"Risk preferences: lambda_s={CONFIG['lambda_s']}, lambda_b={CONFIG['lambda_b']}")
print(f"Scenario name    : {CONFIG['scenario_name']}")
print(f"Save replication FE: {CONFIG['save_replication_fe']}")
print(f"Save best PPA FE : {CONFIG['save_best_fe_even_if_infeasible']}")
print("Decision search  : exact_global_grid")
print(
    "Exact batch sizes:"
    f" nonfix_strike={CONFIG['exact_search_nonfix_strike_chunk_size']},"
    f" fix_volume={CONFIG['exact_search_fix_volume_chunk_size']},"
    f" fix_strike={CONFIG['exact_search_fix_strike_chunk_size']}"
)

summary_rows = []
best_solution_rows = []
best_ppa_solution_rows = []
replication_fe_index_rows = []
run_status_rows = []

for counter, match_id in enumerate(match_ids, start=1):
    match_output_dir = output_root / "Per_Match" / f"Match_{int(match_id):04d}"
    expected_best_solution = match_output_dir / "Best_Solution.csv"
    expected_case_info = match_output_dir / "Case_Info.csv"
    expected_replication_fe = output_root / "Replication_FE" / f"Replication_FE_Match_{int(match_id):04d}__{CONFIG['scenario_name']}.csv"

    can_skip = expected_best_solution.exists() and expected_case_info.exists()
    if CONFIG["save_replication_fe"]:
        can_skip = can_skip and expected_replication_fe.exists()

    if CONFIG["skip_existing_output"] and can_skip:
        print(f"[{counter}/{len(match_ids)}] match_id={match_id}: skipped_existing_output")
        run_status_rows.append({
            "match_id": int(match_id),
            "status": "skipped_existing_output",
            "error_message": "",
        })
        continue

    print(f"[{counter}/{len(match_ids)}] Running match_id={match_id} ...")
    try:
        out = run_one_match_exact_global(
            match_id=int(match_id),
            match_dir=match_dir,
            sample_root=sample_root,
            output_root=output_root,
            match_table_df=match_table_df,
            corr_targets=corr_targets_df,
            config=CONFIG,
        )

        summary_rows.append(out["summary_row"])
        best_solution_rows.append(out["best_solution_row"])
        if out["best_ppa_solution_row"]:
            best_ppa_solution_rows.append(out["best_ppa_solution_row"])
        if out["replication_fe_index_row"] is not None:
            replication_fe_index_rows.append(out["replication_fe_index_row"])

        run_status_rows.append({
            "match_id": int(match_id),
            "status": "success",
            "error_message": "",
            "decision_search_mode": out["search_report"].get("decision_search_mode"),
            "global_optimality_on_exact_grid": out["search_report"].get("global_optimality_on_exact_grid"),
            "n_exact_candidates_evaluated": out["search_report"].get("n_exact_candidates_evaluated"),
            "n_feasible_candidates": out["search_report"].get("n_feasible_candidates"),
            "exact_search_elapsed_seconds": out["search_report"].get("exact_search_elapsed_seconds"),
        })

        print(
            "    done | "
            f"Exact={out['summary_row']['ppa_type']}/{out['summary_row']['profile_type']} | "
            f"Strike={out['summary_row']['strike_price_mwh']} | "
            f"Volume={out['summary_row']['volume_mw']} | "
            f"Candidates={out['summary_row']['n_total_candidates']} | "
            f"Elapsed={out['summary_row']['exact_search_elapsed_seconds']:.1f}s"
        )

    except Exception as exc:
        run_status_rows.append({
            "match_id": int(match_id),
            "status": "failed",
            "error_message": str(exc),
        })
        print(f"    failed: {exc}")

summary_df = pd.DataFrame(summary_rows).sort_values("match_id").reset_index(drop=True) if summary_rows else pd.DataFrame()
best_solution_df = pd.DataFrame(best_solution_rows).sort_values("match_id").reset_index(drop=True) if best_solution_rows else pd.DataFrame()
best_ppa_solution_df = pd.DataFrame(best_ppa_solution_rows).sort_values("match_id").reset_index(drop=True) if best_ppa_solution_rows else pd.DataFrame()
replication_fe_index_df = pd.DataFrame(replication_fe_index_rows).sort_values(["match_id", "scenario_name"]).reset_index(drop=True) if replication_fe_index_rows else pd.DataFrame()
run_status_df = pd.DataFrame(run_status_rows).sort_values("match_id").reset_index(drop=True) if run_status_rows else pd.DataFrame()

summary_path = output_root / "Simulation_Plot_Data_All_Matches.csv"
best_solution_path = output_root / "Simulation_Best_Solutions_All_Matches.csv"
best_ppa_solution_path = output_root / "Simulation_Best_PPA_Solutions_All_Matches.csv"
replication_fe_index_path = output_root / "Replication_FE_Index.csv"
run_status_path = output_root / "Simulation_Run_Status.csv"
config_path = output_root / "Simulation_Config.json"

summary_df.to_csv(summary_path, index=False)
best_solution_df.to_csv(best_solution_path, index=False)
best_ppa_solution_df.to_csv(best_ppa_solution_path, index=False)
replication_fe_index_df.to_csv(replication_fe_index_path, index=False)
run_status_df.to_csv(run_status_path, index=False)

json_ready_config = dict(CONFIG)
json_ready_config["strike_prices"] = [float(x) for x in CONFIG["strike_prices"]]
json_ready_config["fixed_volumes"] = [float(x) for x in CONFIG["fixed_volumes"]]
with open(config_path, "w", encoding="utf-8") as f:
    json.dump(json_ready_config, f, indent=2)

print()
print("Saved output files:")
print("  Plot-ready summary :", summary_path)
print("  Best solutions     :", best_solution_path)
print("  Best PPA solutions :", best_ppa_solution_path)
print("  Replication FE index:", replication_fe_index_path)
print("  Run status         :", run_status_path)
print("  Config             :", config_path)

if not run_status_df.empty:
    print()
    print("Run-status counts:")
    print(run_status_df["status"].value_counts(dropna=False))

summary_df.head()